In [3]:
import numpy as np
import matplotlib.pyplot as plt
from Utilities import extractor
import uproot
import awkward as ak    

x_MH25=extractor("/home/riccardo/Tesi/Cartella_Analisi_Dati/Dati/Tprime_tAq_1800_MH25_LH_2017.root", "Events")


file=uproot.open("/home/riccardo/Tesi/Cartella_Analisi_Dati/Dati/Tprime_tAq_1800_MH25_LH_2017.root")
tree=file["Events"]
booleans= tree.arrays(["FatJet_isMatchedWithA"], library="ak")
Fatjet_isMatchedWithA= booleans["FatJet_isMatchedWithA"]

#Filtriamo i dati

mask = ak.flatten(Fatjet_isMatchedWithA) == 1
x_filtered = x_MH25[mask]



In [4]:
from scipy.special import voigt_profile
from scipy.optimize import curve_fit
from iminuit import Minuit
from iminuit.cost import LeastSquares

x_plot=list(x_filtered)
x_plot.sort()

x_easy=[x for x in x_plot if 0<x<100]

def voigt(x, norm, mu, sigma, gamma):
    return voigt_profile(x-mu, sigma, gamma) * norm

bin_counts, bin_edges = np.histogram(x_easy, bins=50)
bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
bin_width = bin_edges[1] - bin_edges[0]
bin_densities = bin_counts / (len(x_easy) * bin_width)  # Densità normalizzata
yerr=np.sqrt(bin_counts) / (len(x_easy) * bin_width) # Errore standard per i dati binned

ls_voigt=LeastSquares(bin_centers, bin_densities, yerr, model=voigt)

m_voigt=Minuit(ls_voigt,  norm=1, mu=50, sigma=5, gamma=1)
m_voigt.limits["mu"]= (10, 40)
m_voigt.limits["sigma"]= (0.1, 20)
m_voigt.limits["gamma"]= (0.01, 10)
m_voigt.migrad()








┌─────────────────────────────────────────────────────────────────────────┐
│                                Migrad                                   │
├──────────────────────────────────┬──────────────────────────────────────┤
│ FCN = 2750 (χ²/ndof = 59.8)      │              Nfcn = 166              │
│ EDM = 4.63e-05 (Goal: 0.0002)    │            time = 0.7 sec            │
├──────────────────────────────────┼──────────────────────────────────────┤
│          Valid Minimum           │   Below EDM threshold (goal x 10)    │
├──────────────────────────────────┼──────────────────────────────────────┤
│      No parameters at limit      │           Below call limit           │
├──────────────────────────────────┼──────────────────────────────────────┤
│             Hesse ok             │         Covariance accurate          │
└──────────────────────────────────┴──────────────────────────────────────┘
┌───┬───────┬───────────┬───────────┬────────────┬────────────┬─────────┬─────────┬───────┐
│   │ Name  │   Value   │ Hesse Err │ Minos Err- │ Minos Err+ │ Limit-  │ Limit+  │ Fixed │
├───┼───────┼───────────┼───────────┼────────────┼────────────┼─────────┼─────────┼───────┤
│ 0 │ norm  │   0.954   │   0.004   │            │            │         │         │       │
│ 1 │ mu    │  25.386   │   0.015   │            │            │   10    │   40    │       │
│ 2 │ sigma │   2.774   │   0.014   │            │            │   0.1   │   20    │       │
│ 3 │ gamma │   0.201   │   0.008   │            │            │  0.01   │   10    │       │
└───┴───────┴───────────┴───────────┴────────────┴────────────┴─────────┴─────────┴───────┘
┌───────┬─────────────────────────────────────────┐
│       │      norm        mu     sigma     gamma │
├───────┼─────────────────────────────────────────┤
│  norm │  1.72e-05         0 -0.001e-3  0.001e-3 │
│    mu │         0  0.000219  -0.09e-3   0.02e-3 │
│ sigma │ -0.001e-3  -0.09e-3  0.000184  -0.06e-3 │
│ gamma │  0.001e-3   0.02e-3  -0.06e-3  6.93e-05 │
└───────┴─────────────────────────────────────────┘

In [5]:
import pickle

with open("fit_results.pkl", "wb") as f:
    pickle.dump(m_voigt.values, f)

print(m_voigt.values)

print(m_voigt.errors)
print(m_voigt.parameters)

<ValueView norm=0.9543014556838152 mu=25.386269452999606 sigma=2.774403498703792 gamma=0.2006061236569035>
<ErrorView norm=0.004148821580776669 mu=0.014792115634639558 sigma=0.013565863200996287 gamma=0.00832285214782491>
('norm', 'mu', 'sigma', 'gamma')


In [7]:
fit_MH25_values={}
fit_MH25_errors={}

fit_values={'MH25': fit_MH25_values,}
fit_errors={'MH25_errors': fit_MH25_errors}



for param in m_voigt.parameters:
    fit_MH25_values[param] = m_voigt.values[param]

for error in m_voigt.parameters:    #Qui non ho capito come fa a capire che deve estarre gli errori 
    fit_MH25_errors[error] = m_voigt.errors[error]

print(fit_MH25_values)
print(fit_MH25_errors)

import json
#QUi sono andato di metodo oragutang, ho deciso di voler fare 2 file separati peer errori e valori 
#Ho tenuto lo stesso quello con tutti i valori, casomai cambiassi idea

with open("fit_results.json", "r") as f:
    results=json.load(f)

with open("fit_values.json", "r") as g:
    values=json.load(g) 

with open("fit_errors.json", "r") as h:
    errors=json.load(h)


results["MH25"]=fit_MH25_values
results["MH25_errors"]=fit_MH25_errors

with open("fit_results.json", "w") as f:
    json.dump(results, f, indent=1)

values["MH25"]=fit_MH25_values
with open("fit_values.json", "w") as g:
    json.dump(values, g, indent=1)  

errors["MH25_errors"]=fit_MH25_errors
with open("fit_errors.json", "w") as h:
    json.dump(errors, h, indent=1)  





{'norm': 0.9543014556838152, 'mu': 25.386269452999606, 'sigma': 2.774403498703792, 'gamma': 0.2006061236569035}
{'norm': 0.004148821580776669, 'mu': 0.014792115634639558, 'sigma': 0.013565863200996287, 'gamma': 0.00832285214782491}
